# Customer Acquisition Funnel Analysis

I originally completed this analysis as a recruitment case using six months of synthetic marketing and outbound data.

The main question was simple: why did more leads and higher spend not translate into more bookings, and where in the process was the strongest opportunity to improve performance?

I used pandas to load the workbook, DuckDB for most of the analysis in SQL, and matplotlib for the charts.


## Setup


In [ ]:
import math

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DATA_FILE = "../data/customer_funnel_dataset.xlsx"

leads = pd.read_excel(DATA_FILE, "Leads")
outbound = pd.read_excel(DATA_FILE, "Outbound")
prospects = pd.read_excel(DATA_FILE, "Prospects")
cost = pd.read_excel(DATA_FILE, "Cost")

print("Leads:", len(leads))
print("Outbound rows:", len(outbound))
print("Prospects:", len(prospects))
print("Cost rows:", len(cost))


In [ ]:
# Convert the date columns once before registering the tables in DuckDB
leads["Lead_Creation_Timestamp"] = pd.to_datetime(leads["Lead_Creation_Timestamp"])

outbound["Call_Start_Timestamp"] = pd.to_datetime(outbound["Call_Start_Timestamp"])
outbound["Call_End_Timestamp"] = pd.to_datetime(outbound["Call_End_Timestamp"])

prospects["Booking_Timestamp"] = pd.to_datetime(prospects["Booking_Timestamp"])
cost["Cost_Date"] = pd.to_datetime(cost["Cost_Date"])

con = duckdb.connect()

con.register("leads", leads)
con.register("outbound", outbound)
con.register("prospects", prospects)
con.register("cost", cost)


## 1. Check the data first

Before looking at performance, I checked the date coverage, keys, joins, missing values, event order and duplicate rows. This ended up mattering because the outbound table contained duplicated calls.


In [ ]:
# Date coverage of the four tables
con.sql("""
SELECT 'leads' AS table_name,
       COUNT(*) AS rows,
       MIN(Lead_Creation_Timestamp)::DATE AS first_date,
       MAX(Lead_Creation_Timestamp)::DATE AS last_date
FROM leads

UNION ALL

SELECT 'outbound',
       COUNT(*),
       MIN(Call_Start_Timestamp)::DATE,
       MAX(Call_Start_Timestamp)::DATE
FROM outbound

UNION ALL

SELECT 'prospects',
       COUNT(*),
       MIN(Booking_Timestamp)::DATE,
       MAX(Booking_Timestamp)::DATE
FROM prospects

UNION ALL

SELECT 'cost',
       COUNT(*),
       MIN(Cost_Date)::DATE,
       MAX(Cost_Date)::DATE
FROM cost
""").df()


Leads run from January to June, while calls and bookings continue into July because late-June leads were still being worked. I keep leads in the month they were created and later check whether the shorter June follow-up window changes the result.


In [ ]:
# Cost should have one row for every day and acquisition channel
con.sql("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT Cost_Date) AS days,
    COUNT(DISTINCT Acquisition_Channel) AS channels,
    COUNT(DISTINCT Cost_Date) * COUNT(DISTINCT Acquisition_Channel) AS expected_rows,
    SUM(CASE WHEN Cost_EUR <= 0 THEN 1 ELSE 0 END) AS non_positive_cost_rows
FROM cost
""").df()


In [ ]:
# Check IDs and whether calls/bookings can be traced back to a lead
con.sql("""
SELECT
    (SELECT COUNT(*) - COUNT(DISTINCT Lead_ID) FROM leads) AS duplicate_lead_ids,
    (SELECT COUNT(*) - COUNT(DISTINCT Prospect_ID) FROM prospects) AS duplicate_prospect_ids,
    (SELECT COUNT(*) - COUNT(DISTINCT Lead_ID) FROM prospects) AS repeated_leads_in_prospects,
    (SELECT COUNT(*) FROM outbound
        WHERE Lead_ID NOT IN (SELECT Lead_ID FROM leads)) AS calls_without_lead,
    (SELECT COUNT(*) FROM prospects
        WHERE Lead_ID NOT IN (SELECT Lead_ID FROM leads)) AS bookings_without_lead
""").df()


In [ ]:
# Missing values across all four sheets
null_report = []

for name, df in [
    ("leads", leads),
    ("outbound", outbound),
    ("prospects", prospects),
    ("cost", cost)
]:
    nulls = df.isna().sum()
    for column, missing in nulls.items():
        if missing > 0:
            null_report.append([name, column, int(missing)])

pd.DataFrame(null_report, columns=["table", "column", "missing_values"])


In [ ]:
# Check that events happen in a possible order
con.sql("""
SELECT
    (SELECT COUNT(*)
     FROM outbound o
     JOIN leads l USING (Lead_ID)
     WHERE o.Call_Start_Timestamp < l.Lead_Creation_Timestamp) AS calls_before_lead,

    (SELECT COUNT(*)
     FROM outbound
     WHERE Call_End_Timestamp < Call_Start_Timestamp) AS calls_ending_before_start,

    (SELECT COUNT(*)
     FROM prospects p
     JOIN leads l USING (Lead_ID)
     WHERE p.Booking_Timestamp < l.Lead_Creation_Timestamp) AS bookings_before_lead,

    (SELECT COUNT(*)
     FROM prospects p
     JOIN outbound o USING (Lead_ID)
     WHERE o.Call_Start_Timestamp > p.Booking_Timestamp) AS calls_after_booking
""").df()


### Duplicate outbound rows

This was the main data-quality issue. The raw outbound table contains 180 exact duplicate rows. Twenty-three of those duplicated rows have a booking flag, which is why the raw outbound booking flags initially exceed the number of prospects.


In [ ]:
# Confirm and remove exact duplicate outbound rows
duplicate_check = con.sql("""
SELECT
    (SELECT COUNT(*) FROM outbound) AS raw_rows,
    (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM outbound)) AS distinct_rows,
    (SELECT SUM(Booking_Flag) FROM outbound) AS raw_booking_flags,
    (SELECT SUM(Booking_Flag) FROM (SELECT DISTINCT * FROM outbound)) AS clean_booking_flags
""").df()

display(duplicate_check)

con.sql("""
CREATE OR REPLACE TABLE outbound_clean AS
SELECT DISTINCT *
FROM outbound
""")

con.sql("""
SELECT
    COUNT(*) AS clean_rows,
    SUM(Booking_Flag) AS clean_booking_flags
FROM outbound_clean
""").df()


I use `outbound_clean` from this point onward. After deduplication, the booking flags match the 6,111 rows in the prospects table.


In [ ]:
# A few field checks that affect how I interpret the data
campaign_check = con.sql("""
SELECT Campaign_Name, Attempt_Ordinal, COUNT(*) AS calls
FROM outbound_clean
GROUP BY Campaign_Name, Attempt_Ordinal
ORDER BY Campaign_Name, Attempt_Ordinal
""").df()

connection_check = con.sql("""
SELECT
    Contact_Outcome,
    COUNT(*) AS calls,
    AVG(Connected_Flag) AS connected_rate
FROM outbound_clean
GROUP BY Contact_Outcome
ORDER BY calls DESC
""").df()

display(campaign_check)
display(connection_check)


`Campaign_Name` is effectively the attempt number under a different label, so I do not use it as a marketing campaign variable. `Connected_Flag` is also completely determined by the contact outcome in this dataset, so I avoid treating the two as separate signals.


## 2. Build one lead-level table

The outbound table has one row per call. For most of the funnel analysis I need one row per lead, with the number of attempts, whether the lead ever connected, and whether a booking happened.


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE outbound_summary AS
SELECT
    Lead_ID,
    MIN(Call_Start_Timestamp) AS first_call,
    MAX(Call_Start_Timestamp) AS last_call,
    COUNT(*) AS attempts,
    MAX(Connected_Flag) AS ever_connected
FROM outbound_clean
GROUP BY Lead_ID
""")

con.sql("""
CREATE OR REPLACE TABLE lead_level AS
SELECT
    l.Lead_ID,
    l.Lead_Creation_Timestamp,
    date_trunc('month', l.Lead_Creation_Timestamp) AS lead_month,
    l.Acquisition_Channel,
    l.Customer_Segment,
    l.Region,
    o.first_call,
    o.last_call,
    o.attempts,
    COALESCE(o.ever_connected, 0) AS ever_connected,
    CASE WHEN p.Lead_ID IS NOT NULL THEN 1 ELSE 0 END AS booked,
    p.Booking_Timestamp,
    CASE
        WHEN o.first_call IS NULL THEN NULL
        ELSE date_diff('minute', l.Lead_Creation_Timestamp, o.first_call) / 60.0
    END AS ttfc_hours
FROM leads l
LEFT JOIN outbound_summary o USING (Lead_ID)
LEFT JOIN prospects p USING (Lead_ID)
""")


In [ ]:
# Overall funnel
con.sql("""
SELECT
    COUNT(*) AS leads,
    SUM(CASE WHEN attempts IS NOT NULL THEN 1 ELSE 0 END) AS called,
    SUM(ever_connected) AS connected,
    SUM(booked) AS booked,
    ROUND(AVG(booked), 4) AS lead_to_booking
FROM lead_level
""").df()


The full funnel is 30,000 leads → 28,591 called → 19,431 connected → 6,111 booked. Overall lead-to-booking conversion is 20.4%.


## 3. Commercial performance


In [ ]:
# Monthly spend, leads and bookings
con.sql("""
CREATE OR REPLACE TABLE monthly_cost AS
SELECT
    date_trunc('month', Cost_Date) AS lead_month,
    SUM(Cost_EUR) AS cost_eur
FROM cost
GROUP BY lead_month
""")

monthly = con.sql("""
SELECT
    l.lead_month,
    COUNT(*) AS leads,
    SUM(l.booked) AS bookings,
    AVG(l.booked) AS conversion_rate,
    c.cost_eur,
    c.cost_eur / COUNT(*) AS cost_per_lead,
    c.cost_eur / SUM(l.booked) AS cost_per_booking
FROM lead_level l
JOIN monthly_cost c USING (lead_month)
GROUP BY l.lead_month, c.cost_eur
ORDER BY l.lead_month
""").df()

monthly


In [ ]:
# Leads increased while bookings stayed roughly flat
fig, ax = plt.subplots(figsize=(7, 4))

x = range(len(monthly))
labels = monthly["lead_month"].dt.strftime("%b")

ax.bar([i - 0.2 for i in x], monthly["leads"], width=0.4, label="Leads")
ax.bar([i + 0.2 for i in x], monthly["bookings"], width=0.4, label="Bookings")

ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("Count")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()


Lead volume increased from 4,600 in January to 5,400 in June, but bookings stayed around 1,000 per month. Cost per lead barely changed, so the increase in cost per booking mainly follows the fall in conversion rather than more expensive leads.


### 3.1 Check whether the conversion decline is real

Lead-to-booking conversion fell from 21.6% in January to 18.5% in June. I checked three possible explanations before treating that as a meaningful change: shorter follow-up time for June leads, a change in channel mix, and ordinary sampling variation.


In [ ]:
# How long does a booked lead usually take to book?
con.sql("""
SELECT
    MAX(date_diff('day', Lead_Creation_Timestamp, Booking_Timestamp)) AS max_days,
    quantile_cont(date_diff('day', Lead_Creation_Timestamp, Booking_Timestamp), 0.99) AS p99_days,
    median(date_diff('day', Lead_Creation_Timestamp, Booking_Timestamp)) AS median_days
FROM lead_level
WHERE booked = 1
""").df()


In [ ]:
# Recalculate monthly conversion using only leads with the full 13-day window
full_window = con.sql("""
SELECT
    lead_month,
    COUNT(*) AS leads,
    SUM(booked) AS bookings,
    AVG(booked) AS conversion_rate
FROM lead_level
WHERE Lead_Creation_Timestamp <= DATE '2025-06-27'
GROUP BY lead_month
ORDER BY lead_month
""").df()

full_window


June conversion is almost unchanged after removing leads without the full observed conversion window, so late-June censoring does not explain the decline.


In [ ]:
# Hold January's channel mix constant and let only within-channel conversion change
constant_mix = con.sql("""
WITH jan_mix AS (
    SELECT
        Acquisition_Channel,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS jan_share
    FROM lead_level
    WHERE lead_month = (SELECT MIN(lead_month) FROM lead_level)
    GROUP BY Acquisition_Channel
),
monthly_channel AS (
    SELECT
        lead_month,
        Acquisition_Channel,
        AVG(booked) AS conversion_rate
    FROM lead_level
    GROUP BY lead_month, Acquisition_Channel
)
SELECT
    m.lead_month,
    SUM(j.jan_share * m.conversion_rate) AS conversion_with_jan_mix
FROM monthly_channel m
JOIN jan_mix j USING (Acquisition_Channel)
GROUP BY m.lead_month
ORDER BY m.lead_month
""").df()

constant_mix


If the channel mix were the main problem, keeping January's mix fixed should remove most of the decline. It does not. The deterioration is happening inside channels rather than simply because the business acquired more leads from weaker channels.


In [ ]:
# Simple two-proportion check for January versus June
jan = monthly.iloc[0]
jun = monthly.iloc[-1]

p1 = jan["bookings"] / jan["leads"]
p2 = jun["bookings"] / jun["leads"]

pooled = (jan["bookings"] + jun["bookings"]) / (jan["leads"] + jun["leads"])
se = math.sqrt(pooled * (1 - pooled) * (1 / jan["leads"] + 1 / jun["leads"]))

difference = p1 - p2
z_score = difference / se

# Two-sided p-value from the normal distribution
p_value = math.erfc(abs(z_score) / math.sqrt(2))

print(f"January conversion: {p1:.1%}")
print(f"June conversion:    {p2:.1%}")
print(f"Difference:         {difference * 100:.2f} percentage points")
print(f"z-score:            {z_score:.2f}")
print(f"two-sided p-value:  {p_value:.4f}")


This is only a simple sampling check, not a causal model, but it supports the conclusion that the January-to-June gap is larger than I would expect from random month-to-month variation alone.


### 3.2 Which part of the funnel got worse?


In [ ]:
stages = con.sql("""
SELECT
    lead_month,
    AVG(CASE WHEN attempts IS NOT NULL THEN 1 ELSE 0 END) * 100 AS called_per_lead,
    SUM(ever_connected) * 100.0
        / SUM(CASE WHEN attempts IS NOT NULL THEN 1 ELSE 0 END) AS connected_per_called,
    SUM(booked) * 100.0 / SUM(ever_connected) AS booked_per_connected,
    AVG(booked) * 100 AS lead_to_booking
FROM lead_level
GROUP BY lead_month
ORDER BY lead_month
""").df()

stages


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

x = range(len(stages))
labels = stages["lead_month"].dt.strftime("%b")

ax.plot(x, stages["called_per_lead"], marker="o", label="Called / leads")
ax.plot(x, stages["connected_per_called"], marker="o", label="Connected / called")
ax.plot(x, stages["booked_per_connected"], marker="o", linewidth=2.5, label="Booked / connected")

ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel("%")
ax.set_ylim(0, 100)
ax.legend(frameon=False, ncol=3, bbox_to_anchor=(0.5, -0.18), loc="upper center")

plt.tight_layout()
plt.show()


In [ ]:
# Estimate how much of the January-to-June decline sits in the final funnel stage
decline = con.sql("""
SELECT
    lead_month,
    AVG(CASE WHEN attempts IS NOT NULL THEN 1 ELSE 0 END) AS called,
    SUM(ever_connected) * 1.0
        / SUM(CASE WHEN attempts IS NOT NULL THEN 1 ELSE 0 END) AS connected,
    SUM(booked) * 1.0 / SUM(ever_connected) AS booked_per_connected,
    AVG(booked) AS overall
FROM lead_level
GROUP BY lead_month
ORDER BY lead_month
""").df()

jan = decline.iloc[0]
jun = decline.iloc[-1]

actual_drop = jan["overall"] - jun["overall"]
counterfactual = jan["called"] * jan["connected"] * jun["booked_per_connected"]
stage3_drop = jan["overall"] - counterfactual

print(f"Actual conversion decline: {actual_drop * 100:.2f} pp")
print(f"Conversion if only the final stage changed: {counterfactual:.1%}")
print(f"Share of decline in final stage: {stage3_drop / actual_drop:.0%}")


Call coverage and connection barely changed. The main deterioration is after connection: bookings per connected lead fell from 33.4% to 29.1%. Roughly 90% of the overall January-to-June conversion decline sits in that final stage.

That changes the business question. The problem is less about reaching leads and more about what happens once an agent actually speaks to them.


### 3.3 Channel performance


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE channel_cost AS
SELECT Acquisition_Channel, SUM(Cost_EUR) AS cost_eur
FROM cost
GROUP BY Acquisition_Channel
""")

channels = con.sql("""
SELECT
    l.Acquisition_Channel,
    COUNT(*) AS leads,
    SUM(l.booked) AS bookings,
    AVG(l.booked) AS conversion_rate,
    c.cost_eur,
    c.cost_eur / COUNT(*) AS cost_per_lead,
    c.cost_eur / SUM(l.booked) AS cost_per_booking
FROM lead_level l
JOIN channel_cost c USING (Acquisition_Channel)
GROUP BY l.Acquisition_Channel, c.cost_eur
ORDER BY cost_per_booking
""").df()

channels


Cost per lead on its own is misleading. A channel can generate cheap leads and still have an expensive booking if those leads convert poorly.

I treat the channel cost comparison carefully because the case documentation says the acquisition costs are synthetic/randomised. It is useful for working through the KPI logic, but not as evidence of real channel economics.


## 4. Lead treatment


In [ ]:
# How calls end
con.sql("""
SELECT
    Contact_Outcome,
    COUNT(*) AS calls,
    AVG(Connected_Flag) AS connected_rate
FROM outbound_clean
GROUP BY Contact_Outcome
ORDER BY calls DESC
""").df()


Almost half of calls end with no answer. The more interesting question is whether later attempts still produce bookings, and where unbooked leads are being dropped.


In [ ]:
# The tempting but misleading comparison: conversion by total number of attempts
con.sql("""
SELECT
    attempts,
    COUNT(*) AS leads,
    SUM(booked) AS bookings,
    AVG(booked) AS conversion_rate
FROM lead_level
WHERE attempts IS NOT NULL
GROUP BY attempts
ORDER BY attempts
""").df()


This comparison makes later attempts look much worse, but it is selected by the outcome: once a lead books, the calls stop. A lead that converts quickly can never enter the five-attempt group.


In [ ]:
# Better descriptive view: booking rate of each individual call
attempts_table = con.sql("""
SELECT
    Attempt_Ordinal,
    COUNT(*) AS calls,
    SUM(Booking_Flag) AS bookings,
    AVG(Booking_Flag) AS booking_rate_per_call
FROM outbound_clean
GROUP BY Attempt_Ordinal
ORDER BY Attempt_Ordinal
""").df()

attempts_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

rates = attempts_table["booking_rate_per_call"] * 100

ax.bar(attempts_table["Attempt_Ordinal"], rates)
ax.set_xlabel("Attempt number")
ax.set_ylabel("Booking rate per call (%)")
ax.set_xticks(attempts_table["Attempt_Ordinal"])

for x, y in zip(attempts_table["Attempt_Ordinal"], rates):
    ax.text(x, y + 0.3, f"{y:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


The per-call rate does not steadily collapse with later attempts. The fifth recorded attempt still books at about the same observed rate as the first.

This does **not** mean an extra fifth call would cause the same booking rate. Leads reaching later attempts were selected to continue, so these are observed rates rather than a treatment effect.


### 4.1 Where do unbooked leads stop?


In [ ]:
con.sql("""
CREATE OR REPLACE TABLE attempts_ranked AS
SELECT
    Lead_ID,
    Contact_Outcome,
    Attempt_Ordinal,
    ROW_NUMBER() OVER (
        PARTITION BY Lead_ID
        ORDER BY Attempt_Ordinal DESC
    ) AS recency_rank
FROM outbound_clean
""")

con.sql("""
CREATE OR REPLACE TABLE last_outcome AS
SELECT
    Lead_ID,
    Contact_Outcome AS final_outcome,
    Attempt_Ordinal AS final_attempt
FROM attempts_ranked
WHERE recency_rank = 1
""")

stopped = con.sql("""
SELECT
    o.final_outcome,
    COUNT(*) AS leads,
    AVG(o.final_attempt) AS avg_attempts_when_stopped
FROM lead_level l
JOIN last_outcome o USING (Lead_ID)
WHERE l.booked = 0
  AND l.attempts < 5
GROUP BY o.final_outcome
ORDER BY leads DESC
""").df()

stopped


The clearest operational group is the 2,386 unbooked leads whose final recorded outcome is **Follow-up required** but who received no later call. Another 8,610 unbooked leads stopped after **No answer** before reaching five attempts.


In [ ]:
# What happens on the next call after each previous outcome?
next_call = con.sql("""
WITH sequence AS (
    SELECT
        Lead_ID,
        Attempt_Ordinal,
        Contact_Outcome,
        LEAD(Booking_Flag) OVER (
            PARTITION BY Lead_ID
            ORDER BY Attempt_Ordinal
        ) AS next_call_booked
    FROM outbound_clean
)
SELECT
    Contact_Outcome AS previous_outcome,
    COUNT(*) AS leads_that_got_another_call,
    AVG(next_call_booked) AS next_call_booking_rate
FROM sequence
WHERE next_call_booked IS NOT NULL
GROUP BY Contact_Outcome
ORDER BY next_call_booking_rate DESC
""").df()

next_call


When a lead marked **Follow-up required** did receive another call, that next call booked 16.0% of the time, compared with 8.4% after **No answer**.

I use this as a prioritisation signal, not as a forecast. The leads that were actually called again may differ from the leads the team stopped calling.


In [ ]:
# Leads that never received a call
never_called = con.sql("""
SELECT
    lead_month,
    COUNT(*) AS never_called
FROM lead_level
WHERE attempts IS NULL
GROUP BY lead_month
ORDER BY lead_month
""").df()

never_called


A total of 1,409 leads were never called, rising from 207 in January to 288 in June. The data does not say whether this came from capacity, routing rules or deliberate prioritisation, so this is something I would take back to the outbound team rather than guessing.


In [ ]:
# Is outbound effort very different across channels?
con.sql("""
SELECT
    Acquisition_Channel,
    COUNT(*) AS leads,
    AVG(booked) AS conversion_rate,
    AVG(COALESCE(attempts, 0)) AS avg_attempts_per_lead
FROM lead_level
GROUP BY Acquisition_Channel
ORDER BY conversion_rate DESC
""").df()


Average attempts per lead are very similar across channels even though channel conversion varies much more. That suggests there may be room to test a more deliberate queueing rule when outbound capacity is tight.

I would prioritise by the previous call outcome first and use channel only as a secondary signal, then measure the result rather than assuming the historical rates will repeat.


## 5. Recommendations

**1. Test one more attempt for leads who asked to be called back.**  
The 2,386 unbooked leads ending on `Follow-up required` are the clearest group to test. I would measure incremental bookings per 100 extra calls and per hour of agent time.

**2. Make outbound prioritisation more explicit when capacity is tight.**  
Previous call outcome appears more informative than simply giving every channel roughly the same number of attempts. I would test a queue ordered by last outcome first and channel second.

**3. Find out why some paid-for leads never receive a call.**  
The 1,409 never-called leads are an operational gap. Before changing the process, I would confirm whether this is capacity, routing or an intentional rule.

**4. Test media changes rather than making a large budget shift from this dataset alone.**  
Channel differences are useful for diagnosis, but the acquisition costs are synthetic and the data stops at a booked visit. A better decision would also use attended visits, customers, contract value and retention.


## 6. Assumptions and limitations

- The dataset is synthetic. The cost figures and operational timing should not be interpreted as real company performance.
- Leads are assigned to the month in which they were created rather than the month in which they booked.
- The outbound table contained 180 exact duplicate rows, which I removed before the analysis.
- Later-attempt booking rates are descriptive. The leads that received more calls were selected to continue, so the analysis does not identify the causal effect of an additional call.
- The data stops at the booked visit. It does not show whether the visit happened, whether the lead became a customer, contract value, retention or lifetime value.
- There is no agent cost or capacity data, so I cannot tell whether extra outbound activity would be profitable.
- The analysis identifies where the conversion decline appears in the funnel, but it cannot explain why bookings per connected lead fell. I would want information on targeting, creative, bidding, competitors and agent-level operations before drawing a stronger conclusion.
